Source for TF-IDF:
https://towardsdatascience.com/measure-text-weight-using-tf-idf-in-python-plain-code-and-scikit-learn-50cb1e4375ad/

In [ ]:
!wget -nc https://github.com/TurkuNLP/intro-to-nlp/raw/master/Data/imdb_train.json

In [ ]:
import json # JSON encoder and decoder: store python data structures (e.g. lists and dictionaries) as strings

with open("imdb_train.json", "rt", encoding="utf-8") as f:
    data = json.load(f)

print("Data type:", type(data))
print("First item type:", type(data[0]))
print("First item:", data[0])

In [ ]:
import tqdm
from collections import Counter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
token_counter = Counter()

for doc in tqdm.tqdm(data[:1000]): # IMDB documents
    tokenized = tokenizer.tokenize(doc["text"])
    token_counter.update(tokenized)

print("Number of tokens in total:", token_counter.total())
print("Most common tokens:")
for item in token_counter.most_common(20):
  print(item)
print("Vocabulary size:", len(token_counter))

IDF weight function:

In [ ]:
import math

def calculateIDF(tokenInDocument: int, documentInt: int):
    if tokenInDocument == 0:
        return 0
    
    return float(math.log(documentInt/tokenInDocument))

Getting the IDF weights for individual tokens:

In [ ]:
import re

idfDict = dict()
tokenInDocument: int = 0

for token in token_counter:
    for doc in tqdm.tqdm(data[:100]):
        if "#" in token:
            modToken = token.replace('#','')
            pattern = re.compile(r"/w*" + modToken)
            if re.search(pattern, doc["text"]) == True:
                tokenInDocument += 1
                break
            else:
                break

        elif token in doc["text"]:
            tokenInDocument += 1
            break
    
    idfDict[token] = float(calculateIDF(tokenInDocument, 100))
    tokenInDocument = 0

print(idfDict)

In [ ]:
def calculateTFIDF(idf, tokenInt, allTokens):
    if tokenInt == 0:
        return 0
    if idf == 0:
        return 0
    
    return float(tokenInt/allTokens*idf)

In [ ]:
token_counter.clear()
alltfIDF = dict()

for doc in tqdm.tqdm(data[:100]):
    tfIDFList = dict()
    tokenized = tokenizer.tokenize(doc["text"])
    token_counter.update(tokenized)
    tokens = token_counter.keys()
    for token in tokens:
        try:
            tfIdf = calculateTFIDF(idfDict.get(token), token_counter.get(token), len(token_counter))
            tfIDFList[token] = tfIdf
        except:
            print(idfDict.get(token))
    
    alltfIDF[doc["text"]] = tfIDFList

print(alltfIDF)